In [30]:
!pip install flask

from flask import Flask, render_template_string, request, redirect, url_for, send_from_directory
import os
import threading
import webbrowser

# 创建 Flask 应用
app = Flask(__name__)
UPLOAD_FOLDER = "uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok = True)

# HTML 模板
UPLOAD_HTML = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Upload File</title>
    <style>
        /* 页面整体样式 */
        body {
            font-family: Arial, sans-serif;
            background-color: #f5f7fa;
            display: flex;
            justify-content: center;   /* 水平居中 */
            align-items: center;       /* 垂直居中 */
            height: 100vh;             /* 占满整个屏幕高度 */
            margin: 0;
        }

        /* 上传框样式 */
        .upload-box {
            background-color: #fff;
            border: 2px solid #e0e0e0;
            border-radius: 10px;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
            padding: 30px 40px;
            text-align: center;
            width: 400px;
        }

        /* 标题样式 */
        h1 {
            color: #333;
            margin-bottom: 20px;
        }

        /* 上传按钮和文件选择样式 */
        input[type="file"] {
            margin: 10px 0;
        }

        input[type="submit"] {
            background-color: #4CAF50;
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 16px;
        }

        input[type="submit"]:hover {
            background-color: #45a049;
        }

    </style>
</head>

<body>
    <div class="upload-box">
        <h1>📤 Upload a File</h1>
        <form action="/upload" method="post" enctype="multipart/form-data">
            <input type="file" name="file" required><br>
            <input type="submit" value="Upload">
        </form>

        {% if message %}
            <p>{{ message }}</p>
        {% endif %}
    </div>
</body>
</html>
"""

PANEL_HTML = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Input Panel</title>
    <style>
        body {
            margin: 0;
            padding: 0;
            font-family: Arial, sans-serif;
            background: #f4f6f9;
            display: flex;
            height: 100vh;
        }

        /* 左侧 PDF 面板 */
        .left-panel {
            flex: 1;
            background: #fff;
            border-right: 2px solid #ddd;
            display: flex;
            justify-content: center;
            align-items: center;
        }

        iframe {
            width: 100%;
            height: 100%;
            border: none;
        }

        /* 右侧输入面板 */
        .right-panel {
            width: 400px;
            padding: 25px;
            background: #ffffff;
            display: flex;
            flex-direction: column;
        }

        h2 {
            margin-top: 0;
            color: #333;
        }

        textarea {
            flex: 1;
            width: 100%;
            padding: 12px;
            border: 1px solid #ccc;
            border-radius: 6px;
            font-size: 16px;
            resize: none;
        }

        button {
            margin-top: 15px;
            padding: 12px;
            font-size: 16px;
            background: #4a90e2;
            color: white;
            border: none;
            border-radius: 6px;
            cursor: pointer;
        }

        button:hover {
            background: #357ab8;
        }
    </style>
</head>

<body>
    <div class="left-panel">
        <iframe src="/upload/{{ filename }}"></iframe>
    </div>

    <div class="right-panel">
        <h2>Ask Anything You Want</h2>
        <form>
            <textarea placeholder="Ask anything"></textarea>
            <button type="submit">Ask</button>
        </form>
    </div>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(UPLOAD_HTML)

@app.route('/upload', methods = ['POST'])
def upload():
    file = request.files.get('file')
    if not file or file.filename == '':
        return "No file selected."
    path = os.path.join(UPLOAD_FOLDER, file.filename)
    file.save(path)
    return redirect(url_for("panel", filename = file.filename))

@app.route("/panel")
def panel():
    filename = request.args.get("filename")
    return render_template_string(PANEL_HTML, filename = filename)

@app.route("/upload/<path:filename>")
def uploaded_file(filename):
    return send_from_directory(UPLOAD_FOLDER, filename)

# 用线程运行 Flask，避免阻塞 Notebook
def run_app():
    app.run(host = "127.0.0.1", port = 5004, debug = False, use_reloader = False)

threading.Thread(target = run_app).start()

# 自动在浏览器打开网页
webbrowser.open("http://127.0.0.1:5004")

True